<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/Generate_uhi_masks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
STEP 2: Generate ground truth UHI classification masks.

WHAT THIS DOES:
- Reads the mean/std you already computed for each year/season (from the CSV)
- Opens each scene's LST band (band 1 of the stacked file)
- Classifies every pixel into one of 4 UHI classes, based on how far that
  pixel's LST is from that scene's own mean, in units of standard deviation:

      Class 3 = Strong UHI     : LST > mean + 1.5*std
      Class 2 = Weak UHI       : mean + 0.5*std  <  LST <= mean + 1.5*std
      Class 1 = No UHI         : mean - 0.5*std  <= LST <= mean + 0.5*std
      Class 0 = Cooling/Cool   : LST < mean - 0.5*std
      Class 255 = No data      : pixel was outside your AOI / nodata

- Saves one single-band classification GeoTIFF per year/season, same grid
  as your inputs, ready to be tiled alongside your stacked input rasters.

WHERE IT READS FROM:
- STACKED_FOLDER: your stack_<year>_<season>.tif files (band 1 = LST)
- STATS_CSV: the lst_scene_stats.csv from the previous script

BEFORE YOU RUN:
- Confirm the paths below match your setup.
"""

import os
import csv
import rasterio
import numpy as np

STACKED_FOLDER = "/content/drive/MyDrive/DATASET/STACKED"
STATS_CSV = "/content/drive/MyDrive/DATASET/lst_scene_stats.csv"
OUTPUT_FOLDER = "/content/drive/MyDrive/DATASET/MASKS"

LST_BAND_INDEX = 1

NODATA_CLASS = 255  # value used for pixels outside the AOI


def load_stats(csv_path):
    """Read the CSV into a lookup: (year, season) -> {mean, std}."""
    stats = {}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (int(row["year"]), row["season"])
            stats[key] = {"mean": float(row["mean"]), "std": float(row["std"])}
    return stats


def classify_scene(lst, nodata, mean, std):
    """
    Turn a raw LST array into a class array using mean/std thresholds.
    Pixels that are nodata (or NaN) are marked with NODATA_CLASS.
    """
    if nodata is not None and not np.isnan(nodata):
        is_nodata = (lst == nodata)
    else:
        is_nodata = np.isnan(lst)

    classes = np.full(lst.shape, NODATA_CLASS, dtype=np.uint8)

    strong_uhi = lst > (mean + 1.5 * std)
    weak_uhi = (lst > (mean + 0.5 * std)) & (lst <= (mean + 1.5 * std))
    no_uhi = (lst >= (mean - 0.5 * std)) & (lst <= (mean + 0.5 * std))
    cooling = lst < (mean - 0.5 * std)

    classes[cooling & ~is_nodata] = 0
    classes[no_uhi & ~is_nodata] = 1
    classes[weak_uhi & ~is_nodata] = 2
    classes[strong_uhi & ~is_nodata] = 3
    # is_nodata pixels stay at NODATA_CLASS (255), already set above

    return classes


def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    stats = load_stats(STATS_CSV)

    if not stats:
        print(f"No rows found in {STATS_CSV} — check the path and re-run Step 1 if needed.")
        return

    for (year, season), scene_stats in stats.items():
        stack_path = os.path.join(STACKED_FOLDER, f"stack_{year}_{season}.tif")
        if not os.path.exists(stack_path):
            print(f"  MISSING: {stack_path} — skipping")
            continue

        with rasterio.open(stack_path) as src:
            lst = src.read(LST_BAND_INDEX).astype("float64")
            nodata = src.nodata
            meta = src.meta.copy()

        classes = classify_scene(lst, nodata, scene_stats["mean"], scene_stats["std"])

        meta.update(count=1, dtype="uint8", nodata=NODATA_CLASS)

        out_path = os.path.join(OUTPUT_FOLDER, f"mask_{year}_{season}.tif")
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(classes, 1)
            dst.set_band_description(1, "UHI_class")

        # quick summary so you can sanity-check class balance
        unique, counts = np.unique(classes, return_counts=True)
        summary = dict(zip(unique.tolist(), counts.tolist()))
        print(f"{year} {season}: {summary}")

    print(f"\nDone. Masks saved to {OUTPUT_FOLDER}")
    print("Classes: 0=Cooling, 1=No UHI, 2=Weak UHI, 3=Strong UHI, 255=No data")


if __name__ == "__main__":
    main()

2017 Summer: {0: 3006007, 1: 4305286, 2: 3412707, 3: 281619, 255: 10120461}
2017 Winter: {0: 3321009, 1: 4169207, 2: 2817665, 3: 697737, 255: 10120462}
2018 Summer: {0: 2945934, 1: 4486498, 2: 3214124, 3: 359063, 255: 10120461}
2018 Winter: {0: 3419679, 1: 3956682, 2: 2939091, 3: 690166, 255: 10120462}
2019 Summer: {0: 3121122, 1: 4056156, 2: 3606094, 3: 222247, 255: 10120461}
2019 Winter: {0: 3192869, 1: 4344730, 2: 2774011, 3: 693803, 255: 10120667}
2020 Summer: {0: 3161224, 1: 4281916, 2: 3039830, 3: 522649, 255: 10120461}
2020 Winter: {0: 3027161, 1: 4478744, 2: 2884242, 3: 614969, 255: 10120964}
2021 Summer: {0: 2880756, 1: 4478704, 2: 3336528, 3: 309631, 255: 10120461}
2021 Winter: {0: 3085639, 1: 4290041, 2: 3059729, 3: 570133, 255: 10120538}
2022 Summer: {0: 2735301, 1: 4809075, 2: 3141442, 3: 319801, 255: 10120461}
2022 Winter: {0: 3108015, 1: 4146717, 2: 3243773, 3: 507096, 255: 10120479}
2023 Summer: {0: 3226381, 1: 4084046, 2: 3175603, 3: 519589, 255: 10120461}
2023 Winter: